# Load Libraries

In [1]:
import MeshSlicer
import trimesh
import open3d as o3d
import plotly.graph_objects as go
import numpy as np
import os
import pyvista as pv

# Load the meshes

In [2]:
thickness = np.load('./geometries/reparation_2/reparation_2_thickness.npz')

In [6]:
# accumulate all thickness values
all_thickness = []
for key in thickness.files:
    all_thickness.extend(thickness[key])

In [19]:
all_thickness = np.array(all_thickness)
min_thickness = np.min(all_thickness)
max_thickness = np.max(all_thickness)
# subdivide the range into 10 bins
bins = np.linspace(min_thickness, max_thickness, 11)
hist, bin_edges = np.histogram(all_thickness, bins=bins)
# create a bar plot with logarighmic y-axis
hist = go.Figure(data=[go.Bar(x=bin_edges[:-1], y=hist, width=np.diff(bin_edges), marker_color='blue')])
hist.update_yaxes(type='log')
hist.update_layout(
    title='Thickness Distribution',
    xaxis_title='Thickness (mm)',
    yaxis_title='Frequency',
    bargap=0.1,
)
hist.show()

In [13]:
hist

array([   17,    14,     4,     7,    12,   166,  3647, 27994,  2282,
          57])

In [2]:
root_to_surfaces_folder = './geometries/'
surface_to_print = 'reparation_2'

surface_mesh = trimesh.load_mesh(f'{root_to_surfaces_folder}{surface_to_print}/{surface_to_print}.stl')
reference_surface = trimesh.load_mesh(f'{root_to_surfaces_folder}{surface_to_print}/reference_surface.stl')

# Load the object and the optimized scalar map H and the optimized number of levelsets

In [3]:
mesh_obj = MeshSlicer.Mesh.load(f'{root_to_surfaces_folder}{surface_to_print}/{surface_to_print}_initial_mesh_objK90.npz')
initial_H = mesh_obj.H # This is the initial H map, it is only used to visualize the initial isosurfaces

H = np.load(f'{root_to_surfaces_folder}{surface_to_print}/{surface_to_print}_H_from_topK90.npy')
n_levelsets = np.load(f'{root_to_surfaces_folder}{surface_to_print}/{surface_to_print}_n_levelsets_from_topK90.npy')
n_levelsets = np.round(n_levelsets).astype(int)

TBB threads 8
bbox_diag_length = 48.6623
ideal_edge_length = 2.43311
stage = 2
eps_input = 0.0486623
eps = 0.0269389
eps_simplification = 0.0215511
eps_coplanar = 4.86623e-05
dd = 0.0324415
dd_simplification = 0.0259532
[2026-01-22 13:23:29.535] [float-tetwild] [info] remove duplicates: 
[2026-01-22 13:23:29.536] [float-tetwild] [info] #v: 4242 -> 4242
[2026-01-22 13:23:29.536] [float-tetwild] [info] #f: 8480 -> 8480
collapsing 0.329403
swapping 0.00694304
[2026-01-22 13:23:29.874] [float-tetwild] [info] remove duplicates: 
[2026-01-22 13:23:29.874] [float-tetwild] [info] #v: 897 -> 897
[2026-01-22 13:23:29.874] [float-tetwild] [info] #f: 1790 -> 1790
[2026-01-22 13:23:29.875] [float-tetwild] [info] #v = 897
[2026-01-22 13:23:29.875] [float-tetwild] [info] #f = 1790
#boundary_e1 = 0
#boundary_e2 = 0
[2026-01-22 13:23:29.880] [float-tetwild] [info] preprocessing 0.344586s
[2026-01-22 13:23:29.880] [float-tetwild] [info] 
[2026-01-22 13:23:29.904] [float-tetwild] [info] #v = 1865
[2026-0

# Actualize the H map to the optimized one

In [4]:
# Reasign the scalar field
mesh_obj.H = H

In [5]:
n_levelsets

67

In [6]:
printing_data = mesh_obj.provide_printing_data(n_levelsets=n_levelsets)

In [7]:
list_of_levelsets, list_of_distances = printing_data

In [8]:
# save the data for the G-code generator
save_folder = f'{root_to_surfaces_folder}{surface_to_print}/'
save_name_isosurfaces = f'{save_folder}{surface_to_print}_isosurfaces.npz'
save_name_thickness = f'{save_folder}{surface_to_print}_thickness.npz'
MeshSlicer.produce_printing_data(mesh_obj, n_levelsets, save_name_isosurfaces, save_name_thickness)

'Data saved as ./geometries/reparation_2/reparation_2_isosurfaces.npz and ./geometries/reparation_2/reparation_2_thickness.npz'

In [9]:
stophere

NameError: name 'stophere' is not defined

stophere

# Now that we have the levelsets, we can see how good this H and number of levelsets are!

The error is measured by vertex, positive values mean overextrusion and negative values mean underextrusion (in the normal direction on that vertex). Units are still in mm!

In [ ]:
mesh_obj.H = initial_H

In [ ]:
# Compute the surface errors by canal surfaces
array_of_errors = mesh_obj.surface_error_by_canal_surfaces(
                                                list_of_levelsets  = list_of_levelsets )

Let's plot the heatmap!

In [ ]:
reference_volume = trimesh.load_mesh(root_to_surfaces_folder + surface_to_print + '/reference_volume.stl')

In [ ]:
# Decide the bounds you are going to consider for the heatmap
bounds = [array_of_errors.min(), array_of_errors[array_of_errors<1000].max()]
bounds_array = np.abs(np.array(bounds))
# lower_bound, upper_bound = 1*np.array([-1,1])*0.3
lower_bound, upper_bound = 1*np.array([-1,1])*np.round(bounds_array.min(), 1)
# Create the heatmap object
heatmap_object = MeshSlicer.Heatmap(surface_mesh,
                            array_of_errors,
                            array_of_errors.min(),
                            array_of_errors.max(),
                            colorscale = 'turbo_r')
# Plot the heatmap
fig = heatmap_object.plotly_figure()
x,y,z = reference_volume.vertices.T
i,j,k = reference_volume.faces.T
fig.add_mesh3d(x=x, y=y, z=z,
                  i=i, j=j, k=k,
                  name='Reference Volume',
                  opacity= 1,
                  color='grey', showlegend=True)
# heatmap_object.to_Rhino('./dinosaur_heatmap1.ply') # Uncomment to export to Rhino

In [ ]:
array_of_errors.min()

-0.13085142

In [ ]:
heatmap_object.to_Rhino('./pumpkin_heatmap.ply') # Uncomment to export to Rhino

# If you feel like, you can plot the levelsets and the canal surfaces to see how it looks!

In [ ]:
fig = go.Figure()
lvl_vertices = []
lvl_faces = []
offset = 0
# in order to plot the levelsets, we need to create a big mesh with all the levelsets
# and then plot it so that it goes faster
for verts, tris in list_of_levelsets :
    lvl_vertices.append(verts)
    lvl_faces.append(tris + offset)
    offset += verts.shape[0]
big_lvl_vertices = np.vstack(lvl_vertices)
big_lvl_faces = np.vstack(lvl_faces)
x,y,z = big_lvl_vertices.T
i,j,k = big_lvl_faces.T
levels = trimesh.Trimesh(vertices=big_lvl_vertices, faces=big_lvl_faces)
fig.add_mesh3d(x=x, y=y, z=z, i=i, j=j, k=k, name = 'Isosurfaces', showlegend=True)
# levels.export('./levels_dinosaur.stl') # Uncomment to export the levelsets to a file

# Now we can plot the canal surfaces
# Get the canal surfaces
# Recall it this will have a lot of points and faces, so it might take a while
# and it will use a lot of memory, so be careful! It might not work on low memory machines.
canal_vertices, canal_faces = mesh_obj.get_various_canal_surfaces(
            list_of_levelsets = list_of_levelsets,
            previous_isosurface = None, n_sides = 6)
x,y,z = canal_vertices.T
i,j,k = canal_faces.T
canals = trimesh.Trimesh(vertices=canal_vertices, faces=canal_faces)

fig.add_mesh3d(x=x, y=y, z=z, i=i, j=j, k=k,
               name = 'Canal Surfaces',
               showlegend = True,
               opacity = 0.7)
# canals.export('./canals_dinosaur.stl') # Uncomment to export the canal surfaces to a file
fig.update_layout(scene=dict(
        aspectmode='data'
    ));

In [ ]:
fig.show()

# Let's show only the first levelset!

In [ ]:
# show only the first levelset
fig = go.Figure()
first_levelset = list_of_levelsets[0]
x,y,z = first_levelset[0].T
i,j,k = first_levelset[1].T
fig.add_mesh3d(x=x, y=y, z=z, i=i, j=j, k=k, name = 'Isosurface', showlegend=True)
fig.update_layout(scene=dict(
        aspectmode='data'
    ));
fig.show()

# The overall looks of that can be seen with get_cover_mesh method

In [ ]:
cover_vertices, cover_faces = mesh_obj.get_cover_mesh(list_of_levelsets=list_of_levelsets)

# You can see the general description of the errors array if you treat it as a pandas series

In [ ]:
import pandas as pd
# create a pandas series with the errors
error_series = pd.Series(array_of_errors)
error_series.describe()

count    4242.000000
mean       -0.012766
std         0.019268
min        -0.130851
25%        -0.022850
50%        -0.006366
75%         0.001970
max         0.013081
dtype: float64

In [ ]:
arrray_of_errors_copy = array_of_errors.copy()
arrray_of_errors_copy = arrray_of_errors_copy[arrray_of_errors_copy<1000]

In [ ]:
error_series_copy = pd.Series(arrray_of_errors_copy)
error_series_copy.describe()

count    4242.000000
mean       -0.012766
std         0.019268
min        -0.130851
25%        -0.022850
50%        -0.006366
75%         0.001970
max         0.013081
dtype: float64

# For rendering purposes, let's get the canal surfaces such that one out of ten in terms of levelset order is red and the rest white

In [ ]:
chose_color = lambda i: [255, 0, 0, 255] if i % 10 == 0 else [255, 255, 255, 150]  # Red if divisible by 10, white otherwise

canals = []
for j, levelset_value in enumerate(various_levelsets[::20]): # let do it every two values to reduce the size
    level_vertices, level_faces = mesh_obj.get_one_levelset(levelset_value=levelset_value)
    # create the mesh
    canal_mesh = trimesh.Trimesh(vertices=level_vertices, faces=level_faces)
    vertex_color = chose_color(j)
    vertex_colors = np.tile(vertex_color, (len(canal_mesh.vertices), 1))
    canal_mesh.visual.vertex_colors = vertex_colors
    canals.append(canal_mesh)

concatenated_canals = trimesh.util.concatenate(canals)



NameError: name 'various_levelsets' is not defined

In [ ]:
from matplotlib import cm  # viridis colormap

# --- prepare the subset we iterate over (every 2nd levelset) ---
step = 2
levels_subset = various_levelsets[::step]
n = len(levels_subset)
viridis = cm.get_cmap('viridis')

def chose_color(j: int) -> list[int]:
    # map j -> t in [0,1]; guard for n==1
    t = 0.0 if n <= 1 else j / (n - 1)
    r, g, b, a = viridis(t)  # floats in [0,1]
    if j % 10 == 0:
        # Highlight every 10th in the corresponding color
        return [int(255 * r), int(255 * g), int(255 * b), 255]
    else:
        # make it white otherwise
        return [255, 255, 255, 150]
    
    

canals = []
for j, levelset_value in enumerate(levels_subset):
    level_vertices, level_faces = mesh_obj.get_one_levelset(levelset_value=levelset_value)
    canal_mesh = trimesh.Trimesh(vertices=level_vertices, faces=level_faces)

    vertex_color = chose_color(j)
    vertex_colors = np.tile(vertex_color, (len(canal_mesh.vertices), 1)).astype(np.uint8)
    canal_mesh.visual.vertex_colors = vertex_colors

    canals.append(canal_mesh)

concatenated_canals = trimesh.util.concatenate(canals)

In [ ]:
concatenated_canals.export('./concatenated_canals_dinosaur.ply')

In [ ]:
# save the data for the G-code generator
save_folder = f'{root_to_surfaces_folder}{surface_to_print}/'
save_name_isosurfaces = f'{save_folder}{surface_to_print}_isosurfaces.npz'
save_name_thickness = f'{save_folder}{surface_to_print}_thickness.npz'
MeshSlicer.produce_printing_data(mesh_obj, n_levelsets, save_name_isosurfaces, save_name_thickness)

'Data saved as ./geometries/reparation/reparation_isosurfaces.npz and ./geometries/reparation/reparation_thickness.npz'

In [ ]:
# read now the data
data_isosurfaces = np.load(save_name_isosurfaces, allow_pickle=True)
data_thickness = np.load(save_name_thickness, allow_pickle=True)

# Reconstruct the list of levelsets from the saved data
# The data is saved as vertices_0, faces_0, vertices_1, faces_1, etc.
list_of_levelsets_loaded = []
i = 0
while f'vertices_{i}' in data_isosurfaces:
    vertices = data_isosurfaces[f'vertices_{i}']
    faces = data_isosurfaces[f'faces_{i}']
    list_of_levelsets_loaded.append((vertices, faces))
    i += 1

# Reconstruct the list of distances (thickness data)
list_of_distances_loaded = []
i = 0
while f'thickness_{i}' in data_thickness:
    thickness = data_thickness[f'thickness_{i}']
    list_of_distances_loaded.append(thickness)
    i += 1

print(f"Loaded {len(list_of_levelsets_loaded)} levelsets")
print(f"Loaded {len(list_of_distances_loaded)} thickness arrays")
print(f"\nFirst levelset has {len(list_of_levelsets_loaded[0][0])} vertices and {len(list_of_levelsets_loaded[0][1])} faces")
print(f"First thickness array has {len(list_of_distances_loaded[0])} values")

Loaded 60 levelsets
Loaded 60 thickness arrays

First levelset has 730 vertices and 1370 faces
First thickness array has 730 values


In [ ]:
# create the trimesh object corresponding to the first levelset
trimesh_levelset_0 = trimesh.Trimesh(vertices=list_of_levelsets_loaded[0][0],
                                      faces=list_of_levelsets_loaded[0][1])
trimesh_levelset_0.show()